**Purpose:** Run the complete 5-condition experiment: C1 → C2 → ChromaDB → DSPy → C3 (RAG+DSPy) → C4 (Feedback) → C5 (VTS + HMI).

**Runtime:** Colab A100 High-RAM
**Model:** Qwen 2.5-coder:32b via Ollama

---
### Pipeline overview
| # | Phase | Script | Est. time |
|---|-------|--------|-----------|
| 1–5 | Setup | Drive, deps, Ollama, repo | 10 min |
| 6 | C1 Baseline | `multi_main.py` | ~4 hrs (500 signals) |
| 7 | C2 Adaptive | `multi_main_adaptive.py` | ~4 hrs (500 signals) |
| 8 | ChromaDB | `rag.aosp_indexer` | 5 min (or 30s from cache) |
| 9 | DSPy Optimiser | `dspy_opt/optimizer.py` | 2–3 hrs (train-size 8) |
| 10 | Restore + Preflight | — | 1 min |
| 11 | Bug fixes | — | instant |
| 12 | C3 RAG+DSPy | `multi_main_rag_dspy.py` | ~4 hrs (500 signals) |
| 13 | C4 Feedback | `multi_main_c4_feedback.py` | ~4 hrs (500 signals) |
| 14 | Reporting | diagnose → rescore → compare | 2 min |
| 15 | Export | — | 1 min |
| 16 | C5 VTS + HMI | `multi_main_c5.py` | ~1 hr |
| 17 | Utilities | — | as needed |

### Changes in v11 (vs v10)
- **C5 updated** — FakeVehicleHardware patch removed; C5 now generates VTS tests + HMI app only
- **VssVehicleHardware** — C3/C4 generate a from-scratch `IVehicleHardware` implementation (no JSON config, no FakeVHAL)
- **AIDL V3 contract enforced** — `_validate_cpp` in C4 checks HIDL violations before clang
- **Hybrid RAG** — `rank-bm25` added; BM25 + dense + cross-encoder reranker in retriever
- **DSPy cpp program** — stale saved program deleted automatically (signature field names changed)
- **C5 no GCS prereqs** — C5 reads C4 output directly; no FakeVehicleHardware.cpp or aosp_dump needed

## 1. Configuration

In [ ]:
!pip install -q chromadb sentence-transformers rank-bm25

In [ ]:
# ── Configuration — edit these paths once ─────────────────────────
import os, time, shutil, glob, subprocess, requests
from pathlib import Path
import chromadb

DRIVE_SRC    = "/content/drive/MyDrive/mse25_thesis"
REPO_URL     = "https://github.com/appdev1307/code-codegen-aosp-llm-based.git"
REPO_DIR     = "/content/code-codegen-aosp-llm-based"
MODEL_NAME   = "qwen2.5-coder:32b"
OLLAMA_LOG   = "/content/ollama.log"

# AOSP source & ChromaDB
AOSP_SRC_DIR = f"{REPO_DIR}/aosp_source"
CHROMA_DB    = f"{REPO_DIR}/rag/chroma_db"
CHROMA_ZIP   = f"{DRIVE_SRC}/chroma_db.zip"

# AOSP version — must match the build tree on GCP
AOSP_TAG     = "android-14.0.0_r75"

# Restore mappings
RESTORE_MAP = {
    "output_c1.zip":          f"{REPO_DIR}/output",
    "output_c2.zip":          f"{REPO_DIR}/output_adaptive",
    "output_c3.zip":          f"{REPO_DIR}/output_rag_dspy",
    "output_c4.zip":          f"{REPO_DIR}/output_c4_feedback",
    "output_c5.zip":          f"{REPO_DIR}/output_c5",
    "dspy_saved_backup.zip":  f"{REPO_DIR}/dspy_opt/saved",
}

# Ollama parallelism — uses more VRAM but speeds up DSPy 2-3x
os.environ["OLLAMA_NUM_PARALLEL"] = "4"

print("✓ Config loaded (OLLAMA_NUM_PARALLEL=4, AOSP_TAG=" + AOSP_TAG + ")")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_SRC).mkdir(parents=True, exist_ok=True)
print(f"✓ Drive mounted — {DRIVE_SRC}")

## 3. System dependencies

In [ ]:
%%capture
!apt-get update -qq
!apt-get install -y -qq clang checkpolicy zstd

for tool in ["clang", "checkpolicy", "zstd"]:
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    print(f"  {tool}: {'✓ ' + path if path else '❌ NOT FOUND'}")

## 4. Ollama setup

In [ ]:
def start_ollama():
    """Install (if needed), start Ollama with parallel support, wait until healthy."""
    if not Path("/usr/local/bin/ollama").exists():
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True)
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(1)
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT,
                     env={**os.environ})
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"✓ Ollama server ready (took {i+1}s, parallel={os.environ.get('OLLAMA_NUM_PARALLEL', '1')})")
                return True
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("❌ Ollama failed to start — check ollama.log")

start_ollama()

In [ ]:
# Pull model (skip if already cached)
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL_NAME.split(":")[0] in result.stdout:
    print(f"✓ Model {MODEL_NAME} already available")
else:
    print(f"Pulling {MODEL_NAME}...")
    !ollama pull {MODEL_NAME}
!ollama list

## 5. Clone & setup repository

In [ ]:
if Path(REPO_DIR).exists():
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "stash"], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR)
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
%%capture
!pip install -q -r requirements.txt
!pip install -q requests pyyaml chromadb sentence-transformers dspy-ai
!pip install -q jinja2 fastapi uvicorn pydantic
!pip install -q optuna
!pip install -q rank-bm25   # hybrid BM25 retrieval (new requirement)
print("✓ Dependencies installed (including rank-bm25, optuna)")

## 6. Run C1 — Baseline
> Vanilla LLM generation with AIDL-only prompts (~30 min).
> Agent prompts now enforce Android 14 AIDL patterns — no HIDL generation.

In [ ]:
start_ollama()
!rm -rf {REPO_DIR}/output
!python multi_main.py

In [ ]:
# Verify C1 generates AIDL (not HIDL) includes
print("=== C1 C++ includes ===")
!head -10 output/hardware/interfaces/automotive/vehicle/impl/*.cpp 2>/dev/null || echo "No C++ file found"
print("\n=== C1 AIDL files ===")
!find output -name "*.aidl" -not -path "*/.llm_draft/*" | head -5

In [ ]:
# Check if scoring ran
!ls output/SCORES*.json 2>/dev/null

# If no scores file, run the scorer manually
!python experiments/analyze_results.py

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/output_c1", "zip", f"{REPO_DIR}/output")
print(f"✓ C1 saved to Drive ({len(list(Path(f'{REPO_DIR}/output').rglob('*')))} files)")

## 7. Run C2 — Adaptive
> Adaptive generation with Thompson Sampling (~30 min).

In [ ]:
!rm -rf {REPO_DIR}/output_adaptive

In [ ]:
start_ollama()
!rm -rf {REPO_DIR}/output_adaptive
!python multi_main_adaptive.py

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/output_c2", "zip", f"{REPO_DIR}/output_adaptive")
print(f"✓ C2 saved to Drive ({len(list(Path(f'{REPO_DIR}/output_adaptive').rglob('*')))} files)")

## 8. Build ChromaDB from AOSP source (or restore from cache)
> **Must run before DSPy.** Pinned to `android-14.0.0_r75` to match build tree.
> HIDL files excluded — AIDL-only corpus.

**Important:** If you previously cached ChromaDB without HIDL filtering or version pinning,
delete the cache first: `!rm -f {DRIVE_SRC}/chroma_db.zip`

In [ ]:
# ── 8a. Try restoring from Drive ──────────────────────────────────
chroma_restored = False
if Path(CHROMA_ZIP).exists():
    print(f"Found cached ChromaDB: {CHROMA_ZIP}")
    target = Path(CHROMA_DB)
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(CHROMA_ZIP, CHROMA_DB)
    chroma_restored = True
    print("✓ ChromaDB restored from Drive cache")
else:
    print("⚠ No cached ChromaDB — will build from AOSP source")

In [ ]:
# ── 8b. Clone AOSP pinned to android-14.0.0_r75 (only if not restored) ─
if not chroma_restored:
    print(f"Cloning AOSP source (shallow, pinned to {AOSP_TAG}, ~300 MB)...\n")
    aosp_repos = [
        ("https://android.googlesource.com/platform/hardware/interfaces", "hardware"),
        ("https://android.googlesource.com/platform/system/sepolicy",     "sepolicy"),
        ("https://android.googlesource.com/platform/packages/services/Car", "car"),
    ]
    Path(AOSP_SRC_DIR).mkdir(parents=True, exist_ok=True)
    for url, name in aosp_repos:
        dest = f"{AOSP_SRC_DIR}/{name}"
        if Path(dest).exists():
            print(f"  ✓ {name} already cloned")
            continue
        print(f"  Cloning {name}...")
        r = subprocess.run(
            ["git", "clone", "--depth=1", "-b", AOSP_TAG, url, dest],
            capture_output=True, text=True
        )
        print(f"  {'✓' if r.returncode == 0 else '❌'} {name}")
    print(f"\n✓ AOSP source ready ({AOSP_TAG})")
else:
    print("Skipping — ChromaDB restored from cache")

In [ ]:
# ── 8c. Index AOSP → ChromaDB (AIDL-only, HIDL excluded) ────────
if not chroma_restored:
    print("Running AOSP indexer (AIDL-only filter)...\n")
    !python -m rag.aosp_indexer --source {AOSP_SRC_DIR} --db {CHROMA_DB} --force
    print("\n✓ Indexing complete")
else:
    print("Skipping — ChromaDB already populated")

In [ ]:
# ── 8d. Verify ChromaDB ──────────────────────────────────────────
client = chromadb.PersistentClient(path=CHROMA_DB)
collections = client.list_collections()
total_chunks = sum(col.count() for col in collections)
print(f"  Collections: {len(collections)}, Total chunks: {total_chunks}")
for col in collections:
    print(f"    → {col.name}: {col.count()}")
assert total_chunks > 0, "❌ ChromaDB is EMPTY!"
print(f"\n✓ ChromaDB verified")
del client

In [ ]:
# ── 8e. Save to Drive ────────────────────────────────────────────
if not chroma_restored and not Path(CHROMA_ZIP).exists():
    shutil.make_archive(CHROMA_ZIP.replace(".zip", ""), "zip", CHROMA_DB)
    print(f"✓ Saved to {CHROMA_ZIP}")
else:
    print("ChromaDB cache exists — skipping save")

## 9. Run DSPy optimiser (MIPROv2)
> Runs **after** ChromaDB so traces include RAG context.
> `train-size 8` for thesis-quality (~2-3 hrs). **Skip if programs already saved.**

In [ ]:
import shutil, glob
from pathlib import Path

# IMPORTANT: delete stale cpp DSPy program.
# ModernCppVehicleHardwareSignature field names changed (domain/properties/aosp_context
# inputs, cpp_impl output). The old saved program returns empty output silently.
cpp_prog = Path(f"{REPO_DIR}/dspy_opt/saved/cpp_program")
if cpp_prog.exists():
    shutil.rmtree(cpp_prog)
    print("✓ Deleted stale cpp_program (signature changed — will re-optimize)")
else:
    print("✓ cpp_program not present (clean)")

# Count remaining programs (expect 11 after deleting cpp)
dspy_count = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
print(f"  DSPy programs on disk: {dspy_count}/11 (cpp will be re-optimized)")

if dspy_count >= 11:
    print("✓ Other DSPy programs exist — running cpp optimization only")
    start_ollama()
    !python dspy_opt/optimizer.py --agents cpp --mipro-auto light --train-size 4 --force
else:
    print("Running full DSPy optimization (all agents)...")
    start_ollama()
    !python dspy_opt/optimizer.py --mipro-auto light --train-size 4 --force

dspy_count = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
print(f"\n✓ DSPy programs: {dspy_count}/12")

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/dspy_saved_backup", "zip", f"{REPO_DIR}/dspy_opt/saved")
print("✓ DSPy programs saved to Drive (includes re-optimized cpp_program)")

## 10. Restore cached outputs & preflight check
> Restores C1/C2/DSPy from Drive if not on disk (e.g. after Colab restart).

In [ ]:
# ── 10a. Restore if missing ───────────────────────────────────────
for zip_name, target_dir in RESTORE_MAP.items():
    src_path = f"{DRIVE_SRC}/{zip_name}"
    target = Path(target_dir)
    if target.exists() and len(list(target.rglob("*"))) > 0:
        print(f"  ✓ {zip_name}: on disk")
        continue
    if not Path(src_path).exists():
        print(f"  ⚠ {zip_name}: not on Drive")
        continue
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(src_path, target_dir)
    print(f"  ✓ {zip_name}: restored")

# Delete stale cpp_program if restored from old backup
# (signature changed — must re-optimize before C3/C4)
cpp_prog = Path(f"{REPO_DIR}/dspy_opt/saved/cpp_program")
if cpp_prog.exists():
    shutil.rmtree(cpp_prog)
    print("  ✓ Deleted stale cpp_program from backup (will re-optimize in section 9)")

dspy_n = len(glob.glob(f'{REPO_DIR}/dspy_opt/saved/*/program.json'))
print(f"\n  DSPy programs: {dspy_n} (cpp deleted — re-optimize in section 9)")

In [ ]:
# ── 10b. Preflight ───────────────────────────────────────────────
errors = []

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=2)
    models = [m["name"] for m in r.json().get("models", [])]
    if any(MODEL_NAME.split(":")[0] in m for m in models):
        print(f"  ✓ Ollama: {MODEL_NAME}")
    else:
        errors.append(f"Model {MODEL_NAME} not found")
except Exception:
    errors.append("Ollama not responding")

try:
    client = chromadb.PersistentClient(path=CHROMA_DB)
    chunks = sum(c.count() for c in client.list_collections())
    del client
    if chunks > 0:
        print(f"  ✓ ChromaDB: {chunks} chunks")
    else:
        errors.append("ChromaDB empty")
except Exception as e:
    errors.append(f"ChromaDB: {e}")

dspy_n = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
cpp_ok = Path(f"{REPO_DIR}/dspy_opt/saved/cpp_program/program.json").exists()
if not cpp_ok:
    print(f"  ⚠ DSPy: {dspy_n} programs (cpp_program missing — run section 9 to re-optimize)")
else:
    print(f"  ✓ DSPy: {dspy_n} programs")

for label, path in [
    ("C1", f"{REPO_DIR}/output"),
    ("C2", f"{REPO_DIR}/output_adaptive"),
    ("C3", f"{REPO_DIR}/output_rag_dspy"),
    ("C4", f"{REPO_DIR}/output_c4_feedback"),
]:
    if Path(path).exists() and len(list(Path(path).rglob("*"))) > 0:
        print(f"  ✓ {label}: present")
    else:
        print(f"  ⚠ {label}: missing (will be generated or restore from Drive)")

for tool in ["clang", "checkpolicy"]:
    p = subprocess.run(["which", tool], capture_output=True, text=True)
    if p.stdout.strip():
        print(f"  ✓ {tool}")
    else:
        errors.append(f"{tool} not found")

if errors:
    print(f"\n❌ PREFLIGHT FAILED:")
    for e in errors: print(f"  • {e}")
    raise RuntimeError("Fix errors above")
else:
    print(f"\n✓ Preflight passed")

## 11. Apply scoring & output fixes
> 1. `lstrip("**/")` → `removeprefix("**/")` — glob bug
> 2. `_clean_output()` on backend sub-modules — markdown fences
> 3. `output_root` on architect agent — wrong directory

In [ ]:
# Fix 1: lstrip → removeprefix
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists(): continue
    src = Path(fpath).read_text()
    if 'pattern.lstrip("**/")' in src:
        Path(fpath).write_text(src.replace('pattern.lstrip("**/")', 'pattern.removeprefix("**/")'))
        print(f"  ✓ {label}: lstrip → removeprefix")
    else:
        print(f"  ✓ {label}: already patched")

# Fix 2: _clean_output() on backend sub-modules
be_path = f"{REPO_DIR}/agents/rag_dspy_backend_agent.py"
if Path(be_path).exists():
    be = Path(be_path).read_text()
    patched = False
    for old, new, tag in [
        ('model_content = getattr(result, "models_code", "") or ""\n                self._write',
         'model_content = getattr(result, "models_code", "") or ""\n                model_content = self._clean_output(model_content)\n                self._write', "model"),
        ('sim_content = getattr(result, "simulator_code", "") or ""\n                self._write',
         'sim_content = getattr(result, "simulator_code", "") or ""\n                sim_content = self._clean_output(sim_content)\n                self._write', "simulator"),
    ]:
        if old in be and f"_clean_output({tag}_content)" not in be:
            be = be.replace(old, new); patched = True
    if patched:
        Path(be_path).write_text(be); print("  ✓ Backend: _clean_output() added")
    else:
        print("  ✓ Backend: already patched")

# Fix 3: output_root on architect agent
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists(): continue
    src = Path(fpath).read_text()
    old = 'agent = RAGDSPyArchitectAgent(**AGENT_CFG)'
    new = 'agent = RAGDSPyArchitectAgent(**AGENT_CFG, output_root=str(OUTPUT_DIR))'
    if old in src:
        Path(fpath).write_text(src.replace(old, new))
        print(f"  ✓ {label}: output_root added")
    else:
        print(f"  ✓ {label}: already patched")

print("\n✓ All fixes applied")

## 12. Run C3 — RAG + DSPy optimised prompts
> AIDL-only RAG corpus pinned to `android-14.0.0_r75` (~20 min).
> Agent prompts enforce AIDL patterns — C++ should use `aidl::` namespace.

In [ ]:
!rm -rf {REPO_DIR}/output_rag_dspy
start_ollama()
!python apply_chroma_fix.py
!python multi_main_rag_dspy.py

In [ ]:
# Verify C3 C++ uses AIDL (not HIDL)
print("=== C3 C++ includes ===")
!head -10 output_rag_dspy/hardware/interfaces/automotive/vehicle/impl/*.cpp 2>/dev/null || echo "No C++ file"
print("\n=== C3 AIDL files ===")
!find output_rag_dspy -name "*.aidl" | head -5

## 13. Run C4 — Feedback loop
> Generate → validate → refine, up to 3 retries (~20 min).

In [ ]:
!git fetch origin
# Reset your local branch to match remote (overwrites everything)
!git reset --hard origin/main
!git pull origin main

In [ ]:
!rm -rf {REPO_DIR}/output_c4_feedback
start_ollama()
!python multi_main_c4_feedback.py

## 14. Reporting & analysis

In [ ]:
!python diagnose_outputs.py
!python rescore_all_conditions.py
!python compare_matched.py
!python analyze_final.py

In [ ]:
print("=" * 80)
print("MATCHED ANALYSIS RESULTS")
print("=" * 80)
!cat experiments/results/matched_analysis.md

## 15. Export & save to Drive

In [ ]:
from google.colab import files

artifacts = {
    "output_c1":     f"{REPO_DIR}/output",
    "output_c2":     f"{REPO_DIR}/output_adaptive",
    "output_c3":     f"{REPO_DIR}/output_rag_dspy",
    "output_c4":     f"{REPO_DIR}/output_c4_feedback",
    "output_c5":     f"{REPO_DIR}/output_c5",
    "dspy_programs": f"{REPO_DIR}/dspy_opt/saved",
}

for name, src_dir in artifacts.items():
    if Path(src_dir).exists():
        shutil.make_archive(f"/content/{name}", "zip", src_dir)
        shutil.copy(f"/content/{name}.zip", f"{DRIVE_SRC}/{name}.zip")
        print(f"  ✓ {name}.zip ({len(list(Path(src_dir).rglob('*')))} files)")
    else:
        print(f"  ⚠ {name}: not found")

shutil.copytree(f"{REPO_DIR}/experiments/results", f"{DRIVE_SRC}/experiments_results", dirs_exist_ok=True)
print("\n✓ All results saved to Drive")

for name in artifacts:
    zip_path = f"/content/{name}.zip"
    if Path(zip_path).exists():
        files.download(zip_path)

## 16. Run C5 — VTS + HMI Generation
> Generates VTS tests and HMI app for the 500 VSS properties.
> Reads C4 output directly — **no GCS download needed**.
> FakeVehicleHardware patching removed; VssVehicleHardware (C3/C4 output) serves properties.

### 16a. Verify C4 output is ready

In [ ]:
# Authenticate GCS
from google.colab import auth
auth.authenticate_user()
print("✓ GCS authenticated")

In [ ]:
# C5 reads C4 output directly — no GCS downloads or AOSP assets needed.
# Verify the required C4 inputs exist before running C5.
from pathlib import Path
import glob

c4_dir = Path(f"{REPO_DIR}/output_c4_feedback")
output_dir = Path(f"{REPO_DIR}/output")

yaml_files = list(c4_dir.glob("SPEC_FROM_VSS_*.yaml")) if c4_dir.exists() else []
plan_candidates = [
    output_dir / "MODULE_PLAN.json",
    c4_dir / "MODULE_PLAN.json",
]
plan_file = next((p for p in plan_candidates if p.exists()), None)

print("=== C5 Prerequisites ===")
if yaml_files:
    print(f"  ✓ YAML spec: {yaml_files[-1].name}")
else:
    print("  ❌ YAML spec not found — run C4 first")

if plan_file:
    print(f"  ✓ MODULE_PLAN: {plan_file}")
else:
    print("  ❌ MODULE_PLAN.json not found — run C4 first")

if yaml_files and plan_file:
    print("\n✓ C5 ready to run")
else:
    print("\n❌ Run C4 first (section 13)")

### 16b. Run C5 pipeline

In [ ]:
!git pull origin main

In [ ]:
!rm -rf {REPO_DIR}/output_c5

start_ollama()
%cd {REPO_DIR}

# C5 reads: output_c4_feedback/SPEC_FROM_VSS_*.yaml + output/MODULE_PLAN.json
# Generates: output_c5/vts/ (VTS tests) + output_c5/hmi_app/
# Step 2 (FakeVHAL patch) is skipped — VssVehicleHardware serves properties directly.
!python multi_main_c5.py

In [ ]:
# Verify C5 outputs
from pathlib import Path
import json

c5_out = Path(f"{REPO_DIR}/output_c5")
print("=== C5 Output Structure ===")
for subdir in ["vts", "hmi_app"]:    # fake_vhal removed
    path = c5_out / subdir
    if path.exists():
        files = list(path.rglob("*"))
        print(f"  ✓ {subdir}/: {len(files)} files")
        for f in sorted(path.iterdir()):
            print(f"      {f.name}")
    else:
        print(f"  ⚠ {subdir}/: NOT FOUND")

# Show scores
results_path = c5_out / "c5_results.json"
if results_path.exists():
    results = json.loads(results_path.read_text())
    print("\n=== C5 Scores ===")
    print(f"  VTS tests : {results.get('vts', {}).get('score', 0):.3f}")
    print(f"  HMI app   : {results.get('hmi_app', {}).get('score', 0):.3f}")
    print(f"  Overall   : {results.get('overall', 0):.3f}")
    print("  (C5 scores reported separately from C1-C4 comparison)")
else:
    print("\n  ⚠ c5_results.json not found")

### 16c. Upload C5 output to GCS (for GCP VM deployment)

In [ ]:
import shutil, subprocess
from pathlib import Path

c5_zip = "/content/output_c5.zip"
c5_dir = f"{REPO_DIR}/output_c5"

if Path(c5_dir).exists():
    shutil.make_archive("/content/output_c5", "zip", c5_dir)
    r = subprocess.run(["gsutil", "cp", c5_zip, f"{GCS_BUCKET}/output_c5.zip"],
                       capture_output=True, text=True)
    print(f"  {'✓' if r.returncode == 0 else '❌'} output_c5.zip → {GCS_BUCKET}")

    # Also save to Drive
    shutil.copy(c5_zip, f"{DRIVE_SRC}/output_c5.zip")
    print(f"  ✓ output_c5.zip → Google Drive")
else:
    print("  ⚠ output_c5/ not found — run C5 pipeline first")

### 16d. GCP VM deployment — deploy VssVehicleHardware + run VTS (run on GCP VM, not Colab)

```bash
# Download C5 output (VTS test source + HMI app)
gsutil cp gs://aosp-thesis-temp/output_c5.zip ~/
unzip ~/output_c5.zip -d ~/output_c5

# Download C4 output (VssVehicleHardware C++ source)
gsutil cp gs://aosp-thesis-temp/output_c4.zip ~/
unzip ~/output_c4.zip -d ~/output_c4

cd ~/aosp-14-auto
source build/envsetup.sh
lunch aosp_cf_x86_64_auto-trunk_staging-userdebug

# ── Build VssVehicleHardware service ──────────────────────────────
mkdir -p hardware/interfaces/automotive/vehicle/aidl/impl/vss_impl/
cp ~/output_c4/hardware/interfaces/automotive/vehicle/aidl/impl/vss_impl/*.{h,cpp} \
   hardware/interfaces/automotive/vehicle/aidl/impl/vss_impl/
cp ~/output_c4/hardware/interfaces/automotive/vehicle/aidl/impl/vss_impl/Android.bp \
   hardware/interfaces/automotive/vehicle/aidl/impl/vss_impl/
m android.hardware.automotive.vehicle@V3-vss-service

# ── Build VTS test ────────────────────────────────────────────────
rm -rf test/vts/vss_vehicle && mkdir -p test/vts/vss_vehicle
cp ~/output_c5/vts/VtsHalAutomotiveVehicleVss.cpp \
   ~/output_c5/vts/VtsHalAutomotiveVehicleVss.xml \
   ~/output_c5/vts/Android.bp \
   test/vts/vss_vehicle/
# Ensure V3 NDK (not V4)
sed -i 's/automotive.vehicle-V4-ndk/automotive.vehicle-V3-ndk/' test/vts/vss_vehicle/Android.bp
mmm test/vts/vss_vehicle

# ── Deploy VssVehicleHardware on running Cuttlefish ───────────────
adb root && adb remount
adb push out/target/product/vsoc_x86_64/vendor/bin/hw/android.hardware.automotive.vehicle@V3-vss-service \
    /vendor/bin/hw/

# Stop stock VHAL (only one can own IVehicle/default)
adb shell stop vendor.vehicle-default
adb shell lshal | grep automotive.vehicle   # confirm stopped

# Start VssVehicleHardware
adb shell /vendor/bin/hw/android.hardware.automotive.vehicle@V3-vss-service &
adb shell lshal | grep automotive.vehicle   # confirm registered
adb shell dumpsys android.hardware.automotive.vehicle.IVehicle/default

# ── Run VTS ──────────────────────────────────────────────────────
adb push out/target/product/vsoc_x86_64/data/nativetest64/vendor/VtsHalAutomotiveVehicleVss \
    /data/local/tmp/
adb shell /data/local/tmp/VtsHalAutomotiveVehicleVss
# Expected: 4 tests pass (ServiceAvailable, VssPropertiesRegistered, AllIdsUnique, AllIdsWellFormed)

# ── Install HMI app ───────────────────────────────────────────────
mkdir -p packages/apps/VssDashboard
cp -r ~/output_c5/hmi_app/src ~/output_c5/hmi_app/AndroidManifest.xml \
      ~/output_c5/hmi_app/Android.bp packages/apps/VssDashboard/
mmm packages/apps/VssDashboard
adb install -r out/target/product/vsoc_x86_64/system/app/VssDashboardApp/VssDashboardApp.apk
adb shell am start -n com.vss.vehicleapp/.MainActivity
```

## 17. Utilities
> Run manually as needed.

In [ ]:
!grep -n "encode_prop_id\|global_idx" /content/code-codegen-aosp-llm-based/multi_main_c5.py

In [ ]:
!grep -n "compiled_ids\[" /content/code-codegen-aosp-llm-based/multi_main_c5.py

In [ ]:
# Verify VTS test contains the expected number of VSS property IDs
!grep -c "0x2" {REPO_DIR}/output_c5/vts/VtsHalAutomotiveVehicleVss.cpp 2>/dev/null \
    && echo "VSS property IDs in VTS" || echo "VTS not generated yet — run C5 first" 

In [ ]:
# Restart Ollama
# start_ollama()

In [ ]:
!grep "enum" dspy_opt/validators.py | head -5

In [ ]:
# Force rescore from scratch
!rm experiments/results/*.json
!python rescore_all_conditions.py
!python compare_matched.py

In [ ]:
# Force delete ChromaDB cache (if you need to re-index with new filters)
# import os; os.remove(f"{DRIVE_SRC}/chroma_db.zip"); print("✓ Cache deleted")
# !rm -rf rag/chroma_db

In [ ]:
# Free disk (~300 MB)
# shutil.rmtree(AOSP_SRC_DIR, ignore_errors=True); print("✓ AOSP source removed")